In [ ]:
# for running the models on a notebook
!sudo apt-get install -y zstd
!pip install ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

!ollama pull deepseek-r1:14b

In [ ]:
import os, json, random, re, time, ollama

# these scripts were run on the kaggle notebooks but these can be changed to whatever
OUTPUTS_DIR     = "/kaggle/input/datasets/ananyasingh2005/outputs"      
TRANSCRIPTS_DIR = "/kaggle/input/datasets/ananyasingh2005/transcripts"
RESULTS_DIR     = "/kaggle/working"           

# define the seeds and temperature used in this experiment 
SEEDS       = [7, 13, 42, 67, 99, 111, 162, 404, 909]
TEMPERATURE = 0.2

In [ ]:
# add the prompt for LLM-as-a-judge
# ── Cell 4: Prompt ────────────────────────────────────────────────
EVAL_PROMPT = """You will be given an LLM output of the extraction of value, value tensions, and consensus points from the provided deliberative transcript.
The definitions of the constructs are as follows:

Value: A stable and abstract principle that guides a person's judgment about what is important.
Value Tension: A situation in which two independently legitimate values come into conflict. Requires quotes from two different speakers invoking conflicting values over the same issue.
Consensus Point: A claim or principle that receives endorsement from a subset of participants. Requires explicit agreement signals between speakers such as repeated endorsement of the same claim or explicit affirmations.

Evaluate the extraction on the following four criteria using a 1-5 Likert scale:

Faithfulness: are quotes verbatim or near-verbatim from the transcript and not hallucinated?
1 - most quotes are hallucinated or unverifiable
5 - all quotes are verbatim or near-verbatim from the transcript

Construct Correctness: do extractions match the definitions and are correctly labeled?
1 - most extractions violate the construct definitions
5 - all extractions correctly match their definitions

Coverage: does the output identify all major constructs present, or does it miss obvious ones?
1 - many obvious constructs are missed
5 - all major constructs are identified

Label Quality: are labels appropriately abstract summaries of the construct rather than paraphrases of the quote?
1 - labels merely restate the quote
5 - labels are concise and accurately named

Transcript:
{transcript}

Extraction to evaluate:
{response}

Respond in the following JSON format only, with no preamble:
{{
    "faithfulness": {{"score": X, "justification": "..."}},
    "construct_correctness": {{"score": X, "justification": "..."}},
    "coverage": {{"score": X, "justification": "..."}},
    "label_quality": {{"score": X, "justification": "..."}}
}}"""

In [ ]:
# function to load transcript 
def load_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return f.read()

# get the transcript number from the name of the file    
def get_transcript_num(output_id):
    num = int(output_id.replace("output_", ""))
    return ((num - 1) // 9) + 1

# call the judge evaluation independently so there is no leakage between prompts
def call_judge(transcript, response, seed):
    prompt = EVAL_PROMPT.format(transcript=transcript, response=response)
    result = ollama.chat(
        model="deepseek-r1:14b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": TEMPERATURE, "seed": seed}
    )
    return result['message']['content']

# parse the scores by removing the thinking text for deepsekk
def parse_scores(response_text):
    try:
        clean = re.sub(r'<think>[\s\S]*?</think>', '', response_text).strip()
        clean = clean.strip("```json").strip("```").strip()
        return json.loads(clean)
    except json.JSONDecodeError:
        try:
            # match on json 
            match = re.search(r'\{[\s\S]*\}', clean)
            if match:
                # if its valid we can load it 
                return json.loads(match.group())
        except json.JSONDecodeError as e:
            print(f"  JSON parse error: {e}")
            return None

In [ ]:
# remove any weird files that are not .txt and parse
output_files = sorted(
    [f for f in os.listdir(OUTPUTS_DIR) if f.endswith(".txt")], key=lambda x: int(x.replace("output_", "").replace(".txt", ""))
)

# remove any non txt files 
transcript_files = sorted([f for f in os.listdir(TRANSCRIPTS_DIR) if f.endswith(".txt")])

In [ ]:
# create a json file for outputs
results_path = os.path.join(RESULTS_DIR, "results_kaggle_2.json")

# arrays to store results
all_results = []
failed      = []
total       = len(output_files) * len(SEEDS)
count       = 0

# go through every seed
for seed in SEEDS:
    print(f"\nSEED {seed}")


    run_files = output_files.copy()
    random.seed(seed)
    # shuffle the outputs to randomize and reduce positional bias
    random.shuffle(run_files)

    # fore every file we have 
    for filename in run_files:
        count += 1
        output_id      = filename.replace(".txt", "")
        transcript_num = get_transcript_num(output_id)
        transcript     = load_file(
            os.path.join(TRANSCRIPTS_DIR, f"transcript_{transcript_num}.txt")
        )

        print(f"  [{count}/{total}] {output_id} → transcript_{transcript_num} (seed={seed})...")

        extraction = load_file(os.path.join(OUTPUTS_DIR, filename))
        # call the judge on teh transcript we are using, with the file and seed
        raw_response = call_judge(transcript, extraction, seed)
        # parse into json 
        scores = parse_scores(raw_response)

        # if we do not get scores then we store as failed
        if scores is None:
            print(f"parse failed for {output_id} seed {seed}")
            failed.append({
                "output_id":      output_id,
                "transcript_num": transcript_num,
                "seed":           seed,
                "reason":         "parse failure"
            })

        # append the result
        all_results.append({
            "output_id":      output_id,
            "transcript_num": transcript_num,
            "seed":           seed,
            "scores":         scores,
            "raw":            raw_response
        })

        # save to json
        with open(results_path, 'w') as f:
            json.dump(all_results, f, indent=2)


print(f"\n{len(all_results)}/{total} completed. Failed: {len(failed)}")

# store failed outputs
if failed:
    with open(os.path.join(RESULTS_DIR, "failed_kaggle.json"), 'w') as f:
        json.dump(failed, f, indent=2)
    print(f"Failed: {len(failed)}")